## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 🛑 Middleware Termination

## Industry Use Case: Transaction Compliance Screening

This notebook demonstrates how middleware can **terminate** the execution pipeline early.

| Feature | FSI Application |
|---------|-----------------|
| **Pre-Termination** | Block prohibited transaction types |
| **Post-Termination** | Rate limit transaction requests |
| **Short-Circuit** | Return immediate compliance responses |

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print("✅ Environment loaded")


In [ ]:
from collections.abc import Awaitable, Callable

from agent_framework import (
    Agent,
    AgentMiddleware,
    AgentContext,
    AgentResponse,
    Message,
    MiddlewareTermination,
    Content,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("✅ All imports loaded")

## Define Transaction Tool Function

In [ ]:
def check_transaction(transaction_id: str) -> str:
    """Check the status of a transaction."""
    return f"Transaction {transaction_id} is approved and completed."

print("✅ Tool defined: check_transaction")

## Termination Middleware Classes

**PreTerminationMiddleware**: Blocks requests with prohibited transaction types BEFORE processing

**PostTerminationMiddleware**: Allows processing but limits total requests (rate limiting)

In [ ]:
class PreTerminationMiddleware(AgentMiddleware):
    """Terminates execution BEFORE agent processing for prohibited terms."""

    def __init__(self, blocked_words: list[str]):
        self.blocked_words = [word.lower() for word in blocked_words]

    async def process(
        self,
        context: AgentContext,
        next: Callable[[], Awaitable[None]],
    ) -> None:
        last_message = context.messages[-1] if context.messages else None
        if last_message and last_message.text:
            query = last_message.text.lower()
            for blocked_word in self.blocked_words:
                if blocked_word in query:
                    print(f"[PreTermination] ⛔ Blocked: '{blocked_word}' detected")
                    context.result = AgentResponse(
                        messages=[
                            Message(
                                role="assistant",
                                contents=[Content.from_text(f"Cannot process requests containing '{blocked_word}'. Please contact compliance.")],
                            )
                        ]
                    )
                    raise MiddlewareTermination()

        await next()


class PostTerminationMiddleware(AgentMiddleware):
    """Allows processing but terminates after max responses (rate limiting)."""

    def __init__(self, max_responses: int = 2):
        self.max_responses = max_responses
        self.response_count = 0

    async def process(
        self,
        context: AgentContext,
        next: Callable[[], Awaitable[None]],
    ) -> None:
        if self.response_count >= self.max_responses:
            print(f"[PostTermination] ⛔ Rate limit reached ({self.max_responses} requests)")
            raise MiddlewareTermination()

        await next()
        self.response_count += 1
        print(f"[PostTermination] Request {self.response_count}/{self.max_responses}")


print("✅ Middleware classes defined")

## Example 1: Pre-termination (Compliance Blocking)

Blocks requests containing prohibited transaction types.

In [ ]:
async def run_pre_termination_example():
    """Pre-termination blocks prohibited requests."""
    print("\n--- Example 1: Pre-termination Middleware ---\n")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        try:
            agent = Agent(
                client=client,
                name="TransactionAgent",
                instructions="You help with transaction inquiries.",
                tools=[check_transaction],
                middleware=[PreTerminationMiddleware(blocked_words=["offshore", "anonymous"])],
            )

            # Normal query
            print("1. Normal query:")
            result = await agent.run("Check transaction TXN-12345")
            print(f"   Agent: {result.text}\n")

            # Blocked query
            print("2. Blocked query (contains 'offshore'):")
            result = await agent.run("How do I set up offshore transfers?")
            print(f"   Agent: {result.text}\n")
        finally:
            await client.client.close()
            await client.project_client.close()

await run_pre_termination_example()


## Example 2: Post-termination (Rate Limiting)

Allows initial requests but terminates after max responses.

In [ ]:
async def run_post_termination_example():
    """Post-termination limits requests after max responses."""
    print("\n--- Example 2: Post-termination Middleware ---\n")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        try:
            agent = Agent(
                client=client,
                name="TransactionAgent",
                instructions="You help with transaction inquiries.",
                tools=[check_transaction],
                middleware=[PostTerminationMiddleware(max_responses=2)],
            )

            # First request (allowed)
            print("1. First request:")
            result = await agent.run("Check transaction TXN-001")
            print(f"   Agent: {result.text if result and result.text else 'No response'}\n")

            # Second request (allowed)
            print("2. Second request:")
            result = await agent.run("Check transaction TXN-002")
            print(f"   Agent: {result.text if result and result.text else 'No response'}\n")

            # Third request (blocked by rate limit)
            print("3. Third request (rate limited):")
            result = await agent.run("Check transaction TXN-003")
            print(f"   Agent: {result.text if result and result.text else 'No response (rate limited)'}\n")
        finally:
            await client.client.close()
            await client.project_client.close()

await run_post_termination_example()


## Key Takeaways

| Technique | When to Use | FSI Example |
|-----------|-------------|-------------|
| `raise MiddlewareTermination()` before `next()` | Block before processing | Prohibited transaction types |
| `await next()` then `raise MiddlewareTermination()` | Allow then limit | Rate limiting |
| Set `context.result` before terminating | Custom response | Compliance messages |

## Next Steps
- **Override Result** (notebook 8) - Modify agent responses
- **Shared State** (notebook 9) - Cross-middleware communication
